# Graph structures

A unique feature to Graph Neural Networks is that these models learn by the enforcement of a spatial structure; a graph. When applied to epidemiological problems, GNNs should have access to meaningful graph structures which encode the (spatial) relations between the different entities modelled. A graph consists of **nodes** (entities) connected by **edges** (representing relationships between them). Edges may be weighted to quantify connection strength. In epidemiological applications, nodes may represent geographic regions such as districts or municipalities, while edges encode relationships relevant to disease transmission—including travel patterns or spatial proximity. In this project, a graph structure $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ represents Germany, with $N = |\mathcal{V}|$ nodes representing the districts and edges $\mathcal{E}$ encoding various forms of spatial connectivity. GNNs employ message-passing mechanisms that enable each node to aggregate information from its connected neighbors, effectively capturing the relational dynamics within the network. This ability to integrate diverse types of information across graph structures makes GNNs particularly promising for epidemiological modeling (Kraemer, 2025)


### Graph Structures used
In this project, multiple graph construction strategies are explored, to capture different aspects of spatial connectivity. Per administrative unit in Germany, albeit *NUTS1*, *NUTS2* or *NUTS3*, we model epidemiological timeseries, representing incidence rates on casenumbers of the respective infectious disease. The following classes of graph structures are studied:

- **Identity graph**:  A baseline graph structure in which each node is only connected to itself. Therefore, for the prediction of the next state of node $i$, only information of its own past state is used.
- **Mesh graph**: A baseline graph structure in which each node is connected to all other nodes by equal weight.
- **Boolean Neighbors**: A graph representing geographical information by connecting every node to the nodes it shares a geographic border with.
- **Gravity-model**: Graphs encoding geographical and socio-demographic information based on the gravity model of spatial interaction. For any pair of nodes $i, j$, edge weights are computed analogously to the gravitational force between two objects. The connection strength is a function of the population size of $i, j$, and the inverse of their distance. Two variations are used, one with $k=3$ (gravity1) and one with $k=7$ (gravity2).
- **Commuter-based**: Graphs representing commuting data made available by the Bundesagentur für Arbeit \cite{Arbeitsagentur}. Two variations are used based on the commuting data for 2024, one with $k=3$ (commuter 1) and one in which each node's connections are kept, so long as there are more than 1 000 daily commuters between them (commuter2).

 In addition, a threshold of $k$ connections per node may be implemented, emphasizing strong connections over many connections.


In [1]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, DataOrchestrator, GraphOrchestrator

disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        = '2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 3
sequence_length = 1
lag_num         = 1


In [2]:
config = EpiConfig(
    disease             = 'influenza',
    data_env_dir        = get_data_env(),
    date_range          = (min_date, max_date),
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = nuts_level,
    log_transform       = ['incidence'],
    split_berlin        = False,
    include_population  = False,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'cases',
    lag_column          = 'incidence',  
    verbose             = 0
    )    
data_orchestrator = DataOrchestrator(config).build()

In [3]:
graphconstruction   = GraphOrchestrator(data_orchestrator=data_orchestrator)

# identity graph
graphconstruction.generate_graph(method = 'identity')
graphconstruction.rename_graph('identity_selfmean', 'identity_graph')

# mesh graph
graphconstruction.generate_graph(method = 'mesh')
graphconstruction.rename_graph('mesh_selfmean', 'mesh_graph')

# Neighbors
#   boolean-self
graphconstruction.generate_graph(method='boolean_neighbors')
graphconstruction.rename_graph('boolean_neighbors_selfmean',            'geographical_neighbors1')
#   boolean-nonself
graphconstruction.generate_graph(method='boolean_neighbors', self_connection='0')
graphconstruction.rename_graph('boolean_neighbors_self0',               'geographical_neighbors2')
#   numerical-self
graphconstruction.generate_graph(method='boolean_neighbors', scaling_method='rowwise')
graphconstruction.rename_graph('boolean_neighbors_selfmean_rowwise',    'geographical_neighbors3')
#   numerical-nonself
graphconstruction.generate_graph(method='boolean_neighbors', scaling_method='rowwise', self_connection='0')
graphconstruction.rename_graph('boolean_neighbors_self0_rowwise',       'geographical_neighbors4')

# Gravity Models
#   sparse - long distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 3,
                                 alpha              = 1,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity1')
#   sparse - long distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 10,
                                 alpha              = 1,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity2')
#   sparse - short distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 3,
                                 alpha              = 2,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity3')
#   dense - short distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 10,
                                 alpha              = 2,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity4')
#   medium-dense - medium-distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 7,
                                 alpha              = 1.5,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity5')

# Commuter
#   static - 2024-1
#   low threshold
graphconstruction.generate_graph(
    method              = 'commuter', 
    self_connection     = '0',
    commuting_threshold = 500,
    scaling_method      = 'rowwise',
    name_addition       = '1'
)
graphconstruction.rename_graph('commuter_1_self0_rowwise', 'static_commuter24_1')
#   static - 2024-2
#   medium threshold
graphconstruction.generate_graph(
    method              = 'commuter', 
    self_connection     = '0',
    commuting_threshold = 1000,
    scaling_method      = 'rowwise',
    name_addition       = '2'
)
graphconstruction.rename_graph('commuter_2_self0_rowwise', 'static_commuter24_2')
#   static - 2024-3
#   high threshold
graphconstruction.generate_graph(
    method              = 'commuter', 
    self_connection     = '0',
    commuting_threshold = 2500,
    scaling_method      = 'rowwise',
    name_addition       = '3'
)
graphconstruction.rename_graph('commuter_3_self0_rowwise', 'static_commuter24_3')
#   static - 2024-4
#   top_k=4
graphconstruction.generate_graph(
    method              = 'commuter', 
    self_connection     = '0',
    commuting_threshold = 1000,
    scaling_method      = 'rowwise',
    name_addition       = '4',
    top_k               = 4
)
graphconstruction.rename_graph('commuter_4_self0_rowwise', 'static_commuter24_4')

    ✓ Graph registered  : identity_selfmean successfully registered
    ✓ Graph registered  : identity_graph successfully registered
    ✓ Graph removed     : identity_selfmean has been deregistered
    ✓ Graph registered  : mesh_selfmean successfully registered
    ✓ Graph registered  : mesh_graph successfully registered
    ✓ Graph removed     : mesh_selfmean has been deregistered
    ✓ Graph registered  : boolean_neighbors_selfmean successfully registered
    ✓ Graph registered  : geographical_neighbors1 successfully registered
    ✓ Graph removed     : boolean_neighbors_selfmean has been deregistered
    ✓ Graph registered  : boolean_neighbors_self0 successfully registered
    ✓ Graph registered  : geographical_neighbors2 successfully registered
    ✓ Graph removed     : boolean_neighbors_self0 has been deregistered
    ✓ Graph registered  : boolean_neighbors_selfmean_rowwise successfully registered
    ✓ Graph registered  : geographical_neighbors3 successfully registered
    ✓ Gra

In [4]:
figure_empty                    = graphconstruction.preview_graph('empty',                                                          title= "Preview Germany NUTS3")
figure_identity_graph           = graphconstruction.preview_graph('identity_graph',             node_idx = 26,  subplots = True,    title= "Preview identity_graph for Hannover")
figure_mesh_graph               = graphconstruction.preview_graph('mesh_graph',                 node_idx = 26,  subplots = True,    title= "Preview mesh_graph for Hannover")

figure_geographical_neighbors1  = graphconstruction.preview_graph('geographical_neighbors1',    node_idx = 26,  subplots = True,    title= "Preview geographical_neighbors1 for Hannover")
figure_geographical_neighbors1  = graphconstruction.preview_graph('geographical_neighbors2',    node_idx = 26,  subplots = True,    title= "Preview geographical_neighbors1 for Hannover")
figure_geographical_neighbors1  = graphconstruction.preview_graph('geographical_neighbors3',    node_idx = 26,  subplots = True,    title= "Preview geographical_neighbors1 for Hannover")
figure_geographical_neighbors1  = graphconstruction.preview_graph('geographical_neighbors4',    node_idx = 26,  subplots = True,    title= "Preview geographical_neighbors1 for Hannover")

figure_gravity1                 = graphconstruction.preview_graph('gravity1',                   node_idx = 26,  subplots = True,    title= "Preview gravity1 for Hannover")
figure_gravity2                 = graphconstruction.preview_graph('gravity2',                   node_idx = 26,  subplots = True,    title= "Preview gravity2 for Hannover")
figure_gravity3                 = graphconstruction.preview_graph('gravity3',                   node_idx = 26,  subplots = True,    title= "Preview gravity3 for Hannover")
figure_gravity4                 = graphconstruction.preview_graph('gravity4',                   node_idx = 26,  subplots = True,    title= "Preview gravity4 for Hannover")
figure_gravity5                 = graphconstruction.preview_graph('gravity5',                   node_idx = 26,  subplots = True,    title= "Preview gravity5 for Hannover")

figure_static_commuter24_1      = graphconstruction.preview_graph('static_commuter24_1',        node_idx = 26,  subplots = True,    title= "Preview static_commuter24_1 for Hannover")
figure_static_commuter24_2      = graphconstruction.preview_graph('static_commuter24_2',        node_idx = 26,  subplots = True,    title= "Preview static_commuter24_2 for Hannover")
figure_static_commuter24_3      = graphconstruction.preview_graph('static_commuter24_3',        node_idx = 26,  subplots = True,    title= "Preview static_commuter24_3 for Hannover")
figure_static_commuter24_4      = graphconstruction.preview_graph('static_commuter24_4',        node_idx = 26,  subplots = True,    title= "Preview static_commuter24_4 for Hannover")

# graphconstruction.save_graphentry('all')